### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

In [24]:
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def get_weather(location:str)->str:
    """
    Useful for getting current weather data for a specific city or town.
    Use this tool whenever the user asks for temperature, rainfall chance, 
    wind speed, or general weather conditions.
    """
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather, DuckDuckGoSearchRun(
    description="Use this for factual queries, current events, or information about people, places, and political figures."
)])

In [25]:
response = model_with_tools.invoke("I live in the capital of Telangana. Please tell me the current weather there and confirm who the Chief Minister is.")

print(response)

content=[] additional_kwargs={'function_call': {'name': 'duckduckgo_search', 'arguments': '{"query": "capital of Telangana"}'}, '__gemini_function_call_thought_signatures__': {'FH2prXLY': 'EjQKMgEMOdbH7TSBwlgMOYxblNHPehH4sfT/Qh87+3LUy9+WRVYHJwV4xMSa51SMPqoeUcCS'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f16ca-bde4-70b3-bc93-a9fe2c5b5e64-0' tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'capital of Telangana'}, 'id': 'FH2prXLY', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 167, 'output_tokens': 20, 'total_tokens': 187, 'input_token_details': {'cache_read': 0}}


### Tool Execution Loops

In [27]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The current weather in Boston is sunny.


In [ ]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston, MA"}'}, '__gemini_function_call_thought_signatures__': {'kdJWcHY5': 'EjQKMgEMOdbHM0Z/sSyQLflIjXgRsEkvdBEaXEtxQm3BRaLLbnCkGXzA2744eJJA47XjBAxg'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f16ca-f711-79f1-8d54-681997fd835d-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston, MA'}, 'id': 'kdJWcHY5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_tokens': 18, 'total_tokens': 170, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content="It's sunny in Boston, MA", name='get_weather', tool_call_id='kdJWcHY5')]